## 1. Environment Setup & Imports
Setting up the Python environment with necessary libraries for data manipulation, machine learning, and visualization.

### 1.1 Importing Data Manipulation Libraries

In [1]:
#import core data manipulation libraries
import pandas as pd
import numpy as np
import warnings
import os

#suppress warnings for cleaner notebook execution
warnings.filterwarnings('ignore')

### 1.2 Importing Machine Learning Libraries

In [2]:
#import scikit-learn preprocessing and metrics
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import classification_report, confusion_matrix, mean_squared_error, r2_score

#import algorithms
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor, IsolationForest
from sklearn.linear_model import LogisticRegression
import xgboost as xgb

#import imblearn for handling target imbalance
import imblearn
from imblearn.over_sampling import SMOTE

### 1.3 Importing Visualization Libraries

In [3]:
#import plotting libraries
import matplotlib.pyplot as plt
import seaborn as sns

#set default plot style
sns.set_theme(style="whitegrid")

---
## 2. Data Loading & Inspection
Loading all six distinct CSV datasets required for the HR management system pipeline.

In [4]:
#Set project directories
BASE_DIR = os.path.abspath("..")   
DATA_DIR = os.path.join(BASE_DIR, "data")
RAW_DIR = os.path.join(DATA_DIR, "raw")
#Ensure the processed folder exists
PROCESSED_DIR = os.path.join(DATA_DIR, "processed")
os.makedirs(PROCESSED_DIR, exist_ok=True)

In [5]:
#data initial inspection
def inspect_data(df):
    print('-'*30 + ' shape ' + '-'*30)
    print(df.shape)

    print('-'*30 + ' info ' + '-'*30)
    print(df.info())

    print('-'*30 + ' sample ' + '-'*30)
    print(df.sample(n=min(7, len(df))))  # avoid error if df has <7 rows

    print('-'*30 + ' describe - numerical columns ' + '-'*30)
    num_cols = df.select_dtypes(include=[np.number])
    if not num_cols.empty:
        print(num_cols.describe())
    else:
        print("No numerical columns found.")

    print('-'*30 + ' describe - categorical columns ' + '-'*30)
    cat_cols = df.select_dtypes(include=['object', 'category'])
    if not cat_cols.empty:
        print(cat_cols.describe())
    else:
        print("No categorical columns found.")

    print('-'*30 + ' duplicate values ' + '-'*30)
    dup_count = df.duplicated().sum()
    dup_percent = (dup_count / len(df)) * 100
    print(f"Duplicate rows: {dup_count} ({dup_percent:.2f}%)")

    print('-'*30 + ' null values ' + '-'*30)
    null_counts = df.isnull().sum()
    null_percent = (df.isnull().mean() * 100).round(2)
    print(pd.DataFrame({'Null Count': null_counts, 'Null %': null_percent}))

### 2.1 Loading Employee Master Dataset

In [6]:
#load employee demographic and master details
master_df = pd.read_csv(RAW_DIR+'/1_Employee_Master.csv')

In [7]:
inspect_data(master_df)

------------------------------ shape ------------------------------
(2000, 6)
------------------------------ info ------------------------------
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 6 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   Employee_ID       2000 non-null   object
 1   Age               2000 non-null   int64 
 2   Department        2000 non-null   object
 3   Gender            2000 non-null   object
 4   Joining_Date      2000 non-null   object
 5   Attrition_Target  2000 non-null   int64 
dtypes: int64(2), object(4)
memory usage: 93.9+ KB
None
------------------------------ sample ------------------------------
     Employee_ID  Age  Department  Gender Joining_Date  Attrition_Target
329     EMP00330   43  Operations    Male   2021-02-21                 1
449     EMP00450   44       Sales  Female   2022-12-20                 0
1156    EMP01157   22  Technology    

### 2.2 Loading Payroll Information Dataset

In [8]:
#load salary and tax deduction data
payroll_df = pd.read_csv(RAW_DIR+'/2_Payroll_Info.csv')

In [9]:
inspect_data(payroll_df)

------------------------------ shape ------------------------------
(2000, 4)
------------------------------ info ------------------------------
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 4 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   Employee_ID      2000 non-null   object
 1   Base_Salary      2000 non-null   int64 
 2   Bonus_Last_Year  2000 non-null   int64 
 3   Tax_Deductions   2000 non-null   int64 
dtypes: int64(3), object(1)
memory usage: 62.6+ KB
None
------------------------------ sample ------------------------------
     Employee_ID  Base_Salary  Bonus_Last_Year  Tax_Deductions
1861    EMP01862        56702             2341            6304
1373    EMP01374        60714            12552           24622
460     EMP00461        84458             2593           23565
1818    EMP01819       124216             5123            9066
1785    EMP01786        96792          

### 2.3 Loading Performance Ratings Dataset

In [10]:
#load historical manager ratings and kpi scores
performance_df = pd.read_csv(RAW_DIR+'/3_Performance_Ratings.csv')

In [11]:
inspect_data(performance_df)

------------------------------ shape ------------------------------
(2000, 4)
------------------------------ info ------------------------------
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 4 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   Employee_ID               2000 non-null   object
 1   Training_Hours_Completed  2000 non-null   int64 
 2   KPI_Achievement_Pct       2000 non-null   int64 
 3   Manager_Rating            2000 non-null   int64 
dtypes: int64(3), object(1)
memory usage: 62.6+ KB
None
------------------------------ sample ------------------------------
     Employee_ID  Training_Hours_Completed  KPI_Achievement_Pct  \
1248    EMP01249                         4                   81   
1422    EMP01423                         9                   95   
524     EMP00525                         2                   52   
960     EMP00961                

### 2.4 Loading Engagement Data Dataset

In [12]:
#load job satisfaction and work-life balance scores
engagement_df = pd.read_csv(RAW_DIR+'/4_Engagement_Data.csv')

In [13]:
inspect_data(engagement_df)

------------------------------ shape ------------------------------
(2000, 3)
------------------------------ info ------------------------------
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 3 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   Employee_ID         2000 non-null   object
 1   Satisfaction_Score  2000 non-null   int64 
 2   Work_Life_Balance   2000 non-null   int64 
dtypes: int64(2), object(1)
memory usage: 47.0+ KB
None
------------------------------ sample ------------------------------
     Employee_ID  Satisfaction_Score  Work_Life_Balance
606     EMP00607                   3                  3
1838    EMP01839                   4                  2
283     EMP00284                   3                  1
354     EMP00355                   1                  3
1728    EMP01729                   3                  2
701     EMP00702                   2            

### 2.5 Loading Leave & Shift Dataset

In [14]:
#load approved leave records
leave_df = pd.read_csv(RAW_DIR+'/5_Leave_Shift_Data.csv')

In [15]:
inspect_data(leave_df)

------------------------------ shape ------------------------------
(1245, 4)
------------------------------ info ------------------------------
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1245 entries, 0 to 1244
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   Employee_ID  1245 non-null   object
 1   Date         1245 non-null   object
 2   Leave_Type   1245 non-null   object
 3   Shift        1245 non-null   object
dtypes: object(4)
memory usage: 39.0+ KB
None
------------------------------ sample ------------------------------
    Employee_ID        Date Leave_Type    Shift
243    EMP00378  2026-01-13     Casual  Morning
403    EMP00638  2026-02-27       Sick  Morning
658    EMP01073  2026-02-27     Casual  Morning
869    EMP01400  2026-03-03     Unpaid  Morning
728    EMP01194  2026-01-06     Casual  Morning
201    EMP00308  2026-03-17     Unpaid  Morning
613    EMP01007  2026-03-23     Unpaid  Morning
----

### 2.6 Loading Raw Attendance Records Dataset

In [16]:
#load transactional check-in and check-out timestamps
attendance_df = pd.read_csv(RAW_DIR+'/6_Attendance_Records.csv')

In [17]:
inspect_data(attendance_df)

------------------------------ shape ------------------------------
(128000, 4)
------------------------------ info ------------------------------
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 128000 entries, 0 to 127999
Data columns (total 4 columns):
 #   Column       Non-Null Count   Dtype 
---  ------       --------------   ----- 
 0   Employee_ID  128000 non-null  object
 1   Date         128000 non-null  object
 2   Check_In     126755 non-null  object
 3   Check_Out    124300 non-null  object
dtypes: object(4)
memory usage: 3.9+ MB
None
------------------------------ sample ------------------------------
       Employee_ID        Date  Check_In Check_Out
112483    EMP01758  2026-02-19  09:11:18  17:12:44
88254     EMP01379  2026-03-30  08:04:54  19:14:33
15343     EMP00240  2026-03-09  08:52:49  18:30:27
104106    EMP01627  2026-03-02  08:39:26  17:12:46
22422     EMP00351  2026-02-02  07:51:32  18:38:24
5055      EMP00079  2026-03-31  09:19:34  18:15:23
8272      EMP00130  

---
## 3. Data Preprocessing & Master Pipeline Execution
In this section, we transform raw transactional logs into aggregated behavioral features and merge all disparate data sources into a single, unified analytical dataset.

### 3.1 Processing Transactional Attendance (Calculating Overtime, Late Arrivals)
Raw timestamps cannot be fed directly into machine learning models. We need to convert them into datetime objects, calculate the total daily hours, flag late arrivals, and aggregate these metrics per employee over the 90-day window.

In [18]:
#Convert Date, Check_In, and Check_Out to proper datetime objects
attendance_df['Date'] = pd.to_datetime(attendance_df['Date'])
attendance_df['Check_In_Time'] = pd.to_datetime(attendance_df['Date'].astype(str) + ' ' + attendance_df['Check_In'], errors='coerce')
attendance_df['Check_Out_Time'] = pd.to_datetime(attendance_df['Date'].astype(str) + ' ' + attendance_df['Check_Out'], errors='coerce')

#Handle the intentionally missing Check_Out times (Impute with 8 hours after Check_In)
missing_checkout = attendance_df['Check_Out_Time'].isna() & attendance_df['Check_In_Time'].notna()
attendance_df.loc[missing_checkout, 'Check_Out_Time'] = attendance_df.loc[missing_checkout, 'Check_In_Time'] + pd.Timedelta(hours=8)

#Calculate total hours worked per day
attendance_df['Hours_Worked'] = (attendance_df['Check_Out_Time'] - attendance_df['Check_In_Time']).dt.total_seconds() / 3600

# Flag Overtime (working more than 9 hours)
attendance_df['Daily_Overtime'] = attendance_df['Hours_Worked'].apply(lambda x: x - 9 if x > 9 else 0)

#Flag Late Arrivals (Checking in after 10:00 AM)
attendance_df['Is_Late'] = attendance_df['Check_In_Time'].dt.hour >= 10

#Aggregate Leave Days from the Leave Dataset
leave_summary = leave_df.groupby('Employee_ID')['Date'].count().reset_index()
leave_summary.rename(columns={'Date': 'Total_Leave_Days'}, inplace=True)

#Aggregate Attendance metrics per employee
attendance_summary = attendance_df.groupby('Employee_ID').agg(
    Total_Overtime_Hours=('Daily_Overtime', 'sum'),
    Total_Late_Days=('Is_Late', 'sum'),
    Total_Working_Days_Attended=('Hours_Worked', 'count')
).reset_index()

#Round Total_Overtime_Hours to integer
attendance_summary['Total_Overtime_Hours'] = attendance_summary['Total_Overtime_Hours'].round().astype(int)

#Merge Leave summary into Attendance summary
attendance_summary = pd.merge(attendance_summary, leave_summary, on='Employee_ID', how='left')
attendance_summary['Total_Leave_Days'].fillna(0, inplace=True)
attendance_summary['Total_Leave_Days']=attendance_summary['Total_Leave_Days'].astype(int)

#Display the aggregated attendance features
attendance_summary.head()


,Employee_ID,Total_Overtime_Hours,Total_Late_Days,Total_Working_Days_Attended,Total_Leave_Days
0,EMP00001,18,1,63,1
1,EMP00002,22,1,63,1
2,EMP00003,16,1,64,0
3,EMP00004,27,3,64,0
4,EMP00005,23,1,62,2


In [19]:
attendance_summary.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 5 columns):
 #   Column                       Non-Null Count  Dtype 
---  ------                       --------------  ----- 
 0   Employee_ID                  2000 non-null   object
 1   Total_Overtime_Hours         2000 non-null   int64 
 2   Total_Late_Days              2000 non-null   int64 
 3   Total_Working_Days_Attended  2000 non-null   int64 
 4   Total_Leave_Days             2000 non-null   int64 
dtypes: int64(4), object(1)
memory usage: 78.2+ KB


### 3.2 Merging Datasets into a Single Final_HR_Dataset
Now that we have exactly one row per employee for their attendance and behavioral metrics, we perform a series of left joins to consolidate all modules (Master, Payroll, Performance, Engagement, and Attendance) into our `final_hr_df`.

In [20]:
#Initialize the final dataframe with the Employee Master data
final_hr_df = master_df.copy()

#Sequentially merge all other dataframes using Employee_ID as the primary key
final_hr_df = pd.merge(final_hr_df, payroll_df, on='Employee_ID', how='left')
final_hr_df = pd.merge(final_hr_df, performance_df, on='Employee_ID', how='left')
final_hr_df = pd.merge(final_hr_df, engagement_df, on='Employee_ID', how='left')
final_hr_df = pd.merge(final_hr_df, attendance_summary, on='Employee_ID', how='left')

#Verify the shape of the newly merged dataset 
#(Rows should match NUM_EMPLOYEES, Columns should be the sum of all features)
print(f"Final Dataset Shape: {final_hr_df.shape}")

#Display a sample of the consolidated dataset
final_hr_df.head()

Final Dataset Shape: (2000, 18)


,Employee_ID,Age,Department,Gender,Joining_Date,Attrition_Target,Base_Salary,Bonus_Last_Year,Tax_Deductions,Training_Hours_Completed,KPI_Achievement_Pct,Manager_Rating,Satisfaction_Score,Work_Life_Balance,Total_Overtime_Hours,Total_Late_Days,Total_Working_Days_Attended,Total_Leave_Days
0,EMP00001,50,Technology,Female,2025-08-09,0,141307,7692,23161,18,110,5,3,3,18,1,63,1
1,EMP00002,36,Technology,Male,2017-11-14,0,134792,12171,18254,0,81,3,3,4,22,1,63,1
2,EMP00003,29,Technology,Male,2022-01-08,1,65651,6398,12380,38,119,5,2,4,16,1,64,0
3,EMP00004,42,Operations,Female,2022-05-03,1,137919,7252,7228,18,115,5,2,5,27,3,64,0
4,EMP00005,40,Marketing,Female,2023-01-16,0,79963,11721,14814,25,105,5,1,3,23,1,62,2


### 3.3 Missing Value Treatment & Data Type Conversions
Before engineering new features, we must ensure all data types are correct (especially dates) and handle any residual missing values that may have propagated during the merge process.

In [21]:
#check initial missing values
print(final_hr_df.isnull().sum())

#convert joining date to datetime object
final_hr_df['Joining_Date'] = pd.to_datetime(final_hr_df['Joining_Date'])

#impute missing numerical values with median
numeric_cols = final_hr_df.select_dtypes(include=[np.number]).columns
final_hr_df[numeric_cols] = final_hr_df[numeric_cols].fillna(final_hr_df[numeric_cols].median())

#impute missing categorical values with mode
cat_cols = final_hr_df.select_dtypes(exclude=[np.number]).columns
for col in cat_cols:
    final_hr_df[col] = final_hr_df[col].fillna(final_hr_df[col].mode()[0])

#verify missing values are resolved
print("remaining nulls:\n", final_hr_df.isnull().sum().max())

Employee_ID                    0
Age                            0
Department                     0
Gender                         0
Joining_Date                   0
Attrition_Target               0
Base_Salary                    0
Bonus_Last_Year                0
Tax_Deductions                 0
Training_Hours_Completed       0
KPI_Achievement_Pct            0
Manager_Rating                 0
Satisfaction_Score             0
Work_Life_Balance              0
Total_Overtime_Hours           0
Total_Late_Days                0
Total_Working_Days_Attended    0
Total_Leave_Days               0
dtype: int64
remaining nulls:
 0


### 3.4 Feature Engineering
Raw data rarely captures the full context of employee behavior. Here, we create derived features such as tenure, absenteeism rate, and compa-ratio (salary competitiveness) to give the machine learning models stronger predictive signals.

In [22]:
#define the end date of our 3-month analysis window
analysis_end_date = pd.to_datetime('2026-03-31')

#calculate company tenure in months
final_hr_df['Tenure_Months'] = (analysis_end_date - final_hr_df['Joining_Date']).dt.days / 30.44

#calculate absenteeism rate assuming approx 65 working days in a 90 day quarter
final_hr_df['Absenteeism_Rate'] = final_hr_df['Total_Leave_Days'] / 65.0

#calculate compa-ratio based on department median salary
dept_median_salary = final_hr_df.groupby('Department')['Base_Salary'].transform('median')
final_hr_df['Compa_Ratio'] = final_hr_df['Base_Salary'] / dept_median_salary

#calculate average daily overtime
final_hr_df['Avg_Daily_Overtime'] = final_hr_df['Total_Overtime_Hours'] / final_hr_df['Total_Working_Days_Attended'].replace(0, 1)

#drop datetime columns that cannot be processed by standard ml models
cols_to_drop = ['Joining_Date']
final_hr_df.drop(columns=cols_to_drop, inplace=True)

#display the engineered dataset
final_hr_df[['Employee_ID', 'Tenure_Months', 'Absenteeism_Rate', 'Compa_Ratio', 'Avg_Daily_Overtime']].head()

,Employee_ID,Tenure_Months,Absenteeism_Rate,Compa_Ratio,Avg_Daily_Overtime
0,EMP00001,7.687254,0.015385,1.436309,0.285714
1,EMP00002,100.492773,0.015385,1.370088,0.349206
2,EMP00003,50.689882,0.000000,0.667307,0.250000
3,EMP00004,46.911958,0.000000,1.497069,0.421875
4,EMP00005,38.436268,0.030769,0.826337,0.370968


In [23]:
#Save final_hr_df to CSV
output_path = os.path.join(PROCESSED_DIR, "final_hr_df_department.csv")
final_hr_df.to_csv(output_path, index=False)

print(f"Saved final_hr_df to: {output_path}")

Saved final_hr_df to: D:\OFFICE\Tasks\AI_based_HR_system\data\processed\final_hr_df_department.csv


### 3.5 Encoding Categorical Variables (One-Hot Encoding)
Machine learning models require numerical input. We will convert our categorical features, such as `Department` and `Gender`, into a binary format using one-hot encoding while dropping the first category to avoid multicollinearity.

In [24]:
#define categorical columns to encode
categorical_cols = ['Department', 'Gender']

#apply one-hot encoding
final_hr_df = pd.get_dummies(final_hr_df, columns=categorical_cols, drop_first=True)

#convert boolean columns from get_dummies to integers
encoded_cols = [col for col in final_hr_df.columns if final_hr_df[col].dtype == 'bool']
final_hr_df[encoded_cols] = final_hr_df[encoded_cols].astype(int)

### 3.6 Final Data Inspection (Shape check, Target class imbalance check)
Before feeding the data into our models, we perform a final sanity check on the dimensions of our dataset and analyze the distribution of our primary target variable (`Attrition_Target`) to determine if techniques like SMOTE are necessary.

In [25]:
#verify final dataset dimensions
print("final dataset shape:", final_hr_df.shape)

#check class distribution for attrition target
print("\nattrition target distribution:")
print(final_hr_df['Attrition_Target'].value_counts())

#check class distribution as percentages
print("\nattrition target distribution (%):")
print(final_hr_df['Attrition_Target'].value_counts(normalize=True) * 100)

final dataset shape: (2000, 26)

attrition target distribution:
Attrition_Target
0    1455
1     545
Name: count, dtype: int64

attrition target distribution (%):
Attrition_Target
0    72.75
1    27.25
Name: proportion, dtype: float64


In [26]:
final_hr_df

,Employee_ID,Age,Attrition_Target,Base_Salary,Bonus_Last_Year,Tax_Deductions,Training_Hours_Completed,KPI_Achievement_Pct,Manager_Rating,Satisfaction_Score,...,Absenteeism_Rate,Compa_Ratio,Avg_Daily_Overtime,Department_HR,Department_Marketing,Department_Operations,Department_Sales,Department_Technology,Gender_Male,Gender_Other
0,EMP00001,50,0,141307,7692,23161,18,110,5,3,...,0.015385,1.436309,0.285714,0,0,0,0,1,0,0
1,EMP00002,36,0,134792,12171,18254,0,81,3,3,...,0.015385,1.370088,0.349206,0,0,0,0,1,1,0
2,EMP00003,29,1,65651,6398,12380,38,119,5,2,...,0.000000,0.667307,0.250000,0,0,0,0,1,1,0
3,EMP00004,42,1,137919,7252,7228,18,115,5,2,...,0.000000,1.497069,0.421875,0,0,1,0,0,0,0
4,EMP00005,40,0,79963,11721,14814,25,105,5,1,...,0.030769,0.826337,0.370968,0,1,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1995,EMP01996,28,0,60092,11225,11734,21,85,3,3,...,0.000000,0.652281,0.390625,0,0,1,0,0,1,0
1996,EMP01997,35,0,128779,70,22464,26,118,5,3,...,0.015385,1.397857,0.301587,0,0,1,0,0,1,0
1997,EMP01998,30,1,88700,7686,11240,2,67,2,2,...,0.000000,0.901588,0.265625,0,0,0,0,1,1,0
1998,EMP01999,36,0,98105,7879,21581,23,112,5,3,...,0.000000,1.064900,0.296875,0,0,1,0,0,0,0


In [27]:
#Save final_hr_df to CSV
output_path = os.path.join(PROCESSED_DIR, "final_hr_df.csv")
final_hr_df.to_csv(output_path, index=False)

print(f"Saved final_hr_df to: {output_path}")

Saved final_hr_df to: D:\OFFICE\Tasks\AI_based_HR_system\data\processed\final_hr_df.csv
